# Lab 0: Environment Check

Open this notebook from the `lab0_1_environment_setup` folder after completing the setup steps in `01_instructions.md`.

This notebook assumes the current working directory is this lab folder.

Before running the checks, make sure you copied `.env.example` to `.env`.

This notebook checks:
- Python and notebook paths
- required Python packages
- `.env` configuration
- Ollama endpoint connectivity
- Graphviz rendering

In [ ]:
from pathlib import Path
import sys

# Purpose of this block: find the repo root so later cells can open `.env` and `src`.
# In this teaching notebook, we assume the current folder is this lab folder,
# so the repo root is one level up.
cwd = Path.cwd().resolve()
repo_root = cwd.parent
env_path = repo_root / '.env'
src_path = repo_root / 'src'

print('Python executable:', sys.executable)
print('Current working directory:', cwd)
print('Detected repo root:', repo_root)
print('Repo .env exists:', env_path.exists())
print('Repo src exists:', src_path.exists())

if not (repo_root / '.env.example').exists():
    raise FileNotFoundError('Expected .env.example at the repo root.')

if not env_path.exists():
    raise FileNotFoundError('Expected .env at the repo root. Copy .env.example to .env first.')

if not src_path.exists():
    raise FileNotFoundError('Expected src/ at the repo root.')

In [ ]:
# Purpose of this block: import the Python packages used in the remaining setup checks.
import json
import requests
from dotenv import dotenv_values
from graphviz import Digraph
from openai import OpenAI

print('Required packages imported successfully.')

In [ ]:
# Purpose of this block: load MODEL and OLLAMA_BASE_URL from `.env`
# so later cells know which model and endpoint to test.
config = dotenv_values(env_path)
model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')

print('MODEL:', model)
print('OLLAMA_BASE_URL:', ollama_base_url)

if not model:
    raise ValueError('MODEL is missing from .env')

if not ollama_base_url:
    raise ValueError('OLLAMA_BASE_URL is missing from .env')

In [ ]:
# Purpose of this block: contact the Ollama server and print the raw JSON response
# so students can observe its structure before extracting values from it.
# `/v1` is the OpenAI-compatible chat API path used later when the notebook asks the model a question.
# `/api/tags` is the Ollama endpoint that returns the list of models installed on the server.
# Example: `http://localhost:11434/v1` should become `http://localhost:11434/api/tags`.
tags_url = ollama_base_url.rstrip('/').replace('/v1', '/api/tags')

# Ask Ollama for the list of installed models.
tags_response = requests.get(tags_url, timeout=10)
tags_response.raise_for_status()

# Read the JSON response so students can inspect it directly.
response_data = tags_response.json()
print('Model list URL:', tags_url)
print('Full JSON response:')
print(json.dumps(response_data, indent=2))


In [ ]:
# Purpose of this block: extract the `models` list from the JSON response,
# build a simple list of model names, and check whether the configured model exists.
models_data = response_data.get('models', [])

available_models = []
for item in models_data:
    model_name = item.get('name')
    if model_name:
        available_models.append(model_name)

print('Top-level keys in the JSON response:', list(response_data.keys()))
if models_data:
    print('Example model record from the response:')
    print(models_data[0])
print('Available models:', available_models)

if model not in available_models:
    raise ValueError(f'Model {model!r} was not found. Update .env or ask your instructor.')

print(f'Model {model!r} is available.')


In [ ]:
# Purpose of this block: send one simple question to the configured model
# so students can observe the chat response object, not just the final reply text.
# Create an OpenAI-compatible client that points to the Ollama server from `.env`.
# `base_url=ollama_base_url` tells the client where the model server is running.
# `api_key='ollama'` is a placeholder value commonly used with a local Ollama server.
client = OpenAI(base_url=ollama_base_url, api_key='ollama')

# `chat.completions.create(...)` sends a chat request to the model.
# `model=model` tells the API which model name from `.env` to use.
# `messages=[...]` provides the conversation input as a list of message dictionaries.
# Each message has a `role` such as `user` or `assistant`, plus a `content` string.
# Here we send one user message that asks for a short, easy-to-check reply.
chat_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            'role': 'user',
            'content': 'If the model connection is working, reply with exactly: Your AI setup is working.'
        }
    ]
)

# The response object can contain one or more candidate answers in `choices`.
# In this notebook we read the first answer with `choices[0]`.
# `message.content` is the assistant's text reply.
assistant_reply = chat_response.choices[0].message.content

print('Raw chat response object:')
print(chat_response)
print('Number of choices returned:', len(chat_response.choices))
print('Assistant reply:')
print(assistant_reply)

In [ ]:
# Purpose of this block: make sure Graphviz can build and display a simple diagram
# in the notebook environment.
diagram = Digraph(comment='Setup Check')
diagram.attr(rankdir='LR')
diagram.node('env', '.env')
diagram.node('ollama', 'Configured Ollama Endpoint')
diagram.node('model', f'Model: {model}')
diagram.node('nb', 'Notebook')
diagram.edges([('env', 'nb'), ('ollama', 'nb'), ('model', 'ollama')])
diagram

## Complete

If all cells run successfully, your environment is ready for Lab 1.